In [1]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.frequent_patterns import association_rules
import pandas as pd
import ast

In [2]:
df_carts = pd.read_csv("../../data/tesco/tesco_carts_clean.csv")
df_inventory = pd.read_csv("../../data/tesco/tesco_inventory_clean.csv")

In [3]:
df_carts.head()

,cart_id,cart
0,0,[73314923]
1,1,"[58175124, 50502269, 70943424, 57346955, 56036..."
2,2,"[50962501, 50503297, 50507984, 50169663, 51965..."
3,3,"[72680530, 62284801, 58098273]"
4,4,"[56373768, 67336474, 52844621, 54921285]"


In [4]:
df_inventory.head()

,product_id,category,description,ingredients,energy,fat,saturates,salt,sugars,protein,carbohydrate,fibre,avg_price
0,68238698,fruit_veg,Tesco Baby Corn 190G (M),no_ingredients,28.0,0.4,0.1,0.3,1.9,2.5,2.7,2.0,1.645
1,53426251,fruit_veg,Tesco Cranberries 100G,Pineapple Juice from Concentrate Cranberries S...,336.0,1.6,0.2,0.1,65.0,0.3,77.4,5.5,1.495
2,59445495,fruit_veg,Tesco Crispy Slices 350G,Potato (78%) Batter Rapeseed Oil Batter contai...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.500
3,74533881,sweets,Kelloggs Pop Tarts Frosted S'mores 416G,"Enriched Flour (Wheat Flour, Niacin, Reduced I...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.500
4,54198489,grains,Osem Bamba Peanut Snack 25G,Peanuts (50%) Corn Palm Oil Salt,534.0,34.0,6.6,1.0,3.2,0.0,0.0,0.0,0.400


To learn more about their customers’ behavior, the Tesco marketing department is interested in three
pieces of information.
- A list of triplets of products (A, B, C) such that C is purchased frequently after A and B. They are
interested in strong associations, such that the likelihood of the association is at least 1.8 times higher
than chance.
- A list of pairs (A, B) such that B is frequently purchased after A, and that the pair (A, B) is found
least in 0.5% of the carts.
- The top three items by confidence in the association rule A → B,

In [5]:
# Inspecting the carts dataset carts that contain more than a single item is a list but the list is in quotation marks which turns it into a string instead of a list
# which I can verify by simply grabbing one cart which multiple items and making python determine its type.
print(type(df_carts['cart'][1]))

<class 'str'>


In [6]:
# To convert the string representation of a list to a list I use ast.literal_eval, inspired from here: https://stackoverflow.com/questions/1894269/how-to-convert-string-representation-of-list-to-a-list
# and from what we did in exercises for lecture 5.
carts = df_carts['cart'].apply(ast.literal_eval).tolist()

In [7]:
# Verify that the same cart as before now actually has type list
print(type(carts[1]))

<class 'list'>


In [8]:
print(len(carts))

1582801


In [9]:
te = TransactionEncoder()
te_data = te.fit(carts).transform(carts, sparse=True)
df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)

# product indices must either start from 0 or be strings
df.columns = [str(i) for i in df.columns] 

In [ ]:
# Very low minimum support because the dataset has 1.5m entries, a value of 0.05 found no triplets
frequent_itemsets = apriori(df, min_support=0.001, use_colnames=True, low_memory=True)

In [24]:
frequent_itemsets

,support,itemsets
0,0.001324,frozenset({50014680})
1,0.001021,frozenset({50017924})
2,0.001182,frozenset({50019277})
3,0.002565,frozenset({50020286})
4,0.001060,frozenset({50020430})
...,...,...
1902,0.001171,"frozenset({62816594, 77454543, 50502436})"
1903,0.001042,"frozenset({75341714, 77454543, 50502436})"
1904,0.001058,"frozenset({50503441, 50550228, 54550994})"
1905,0.001630,"frozenset({50503441, 50550228, 62816594})"


In [34]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=0.0000000001)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({50264481}),frozenset({50502269}),0.007201,0.088677,0.002082,0.289174,3.260977,1.0,0.001444,1.282061,0.698372,0.022201,0.220006,0.156328
1,frozenset({50502269}),frozenset({50264481}),0.088677,0.007201,0.002082,0.023483,3.260977,1.0,0.001444,1.016673,0.760810,0.022201,0.016400,0.156328
2,frozenset({50264481}),frozenset({50502436}),0.007201,0.054489,0.002134,0.296280,5.437444,1.0,0.001741,1.343590,0.822009,0.035824,0.255725,0.167718
3,frozenset({50502436}),frozenset({50264481}),0.054489,0.007201,0.002134,0.039156,5.437444,1.0,0.001741,1.033257,0.863121,0.035824,0.032187,0.167718
4,frozenset({50689433}),frozenset({50264481}),0.032198,0.007201,0.001004,0.031179,4.329787,1.0,0.000772,1.024750,0.794627,0.026147,0.024152,0.085295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1379,"frozenset({54550994, 62816594})",frozenset({50503441}),0.003755,0.036397,0.001044,0.278096,7.640645,1.0,0.000908,1.334807,0.872397,0.026704,0.250828,0.153394
1380,"frozenset({50503441, 62816594})",frozenset({54550994}),0.005358,0.061543,0.001044,0.194906,3.167004,1.0,0.000715,1.165650,0.687930,0.015858,0.142109,0.105938
1381,frozenset({54550994}),"frozenset({50503441, 62816594})",0.061543,0.005358,0.001044,0.016970,3.167004,1.0,0.000715,1.011812,0.729116,0.015858,0.011674,0.105938
1382,frozenset({50503441}),"frozenset({54550994, 62816594})",0.036397,0.003755,0.001044,0.028693,7.640645,1.0,0.000908,1.025675,0.901949,0.026704,0.025032,0.153394


In [36]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,frozenset({50264481}),frozenset({50502269}),0.007201,0.088677,0.002082,0.289174,3.260977,1.0,0.001444,1.282061,0.698372,0.022201,0.220006,0.156328,1
1,frozenset({50502269}),frozenset({50264481}),0.088677,0.007201,0.002082,0.023483,3.260977,1.0,0.001444,1.016673,0.760810,0.022201,0.016400,0.156328,1
2,frozenset({50264481}),frozenset({50502436}),0.007201,0.054489,0.002134,0.296280,5.437444,1.0,0.001741,1.343590,0.822009,0.035824,0.255725,0.167718,1
3,frozenset({50502436}),frozenset({50264481}),0.054489,0.007201,0.002134,0.039156,5.437444,1.0,0.001741,1.033257,0.863121,0.035824,0.032187,0.167718,1
4,frozenset({50689433}),frozenset({50264481}),0.032198,0.007201,0.001004,0.031179,4.329787,1.0,0.000772,1.024750,0.794627,0.026147,0.024152,0.085295,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1379,"frozenset({54550994, 62816594})",frozenset({50503441}),0.003755,0.036397,0.001044,0.278096,7.640645,1.0,0.000908,1.334807,0.872397,0.026704,0.250828,0.153394,2
1380,"frozenset({50503441, 62816594})",frozenset({54550994}),0.005358,0.061543,0.001044,0.194906,3.167004,1.0,0.000715,1.165650,0.687930,0.015858,0.142109,0.105938,2
1381,frozenset({54550994}),"frozenset({50503441, 62816594})",0.061543,0.005358,0.001044,0.016970,3.167004,1.0,0.000715,1.011812,0.729116,0.015858,0.011674,0.105938,1
1382,frozenset({50503441}),"frozenset({54550994, 62816594})",0.036397,0.003755,0.001044,0.028693,7.640645,1.0,0.000908,1.025675,0.901949,0.026704,0.025032,0.153394,1


In [39]:
rules[ (rules['antecedent_len'] >= 2) &
       (rules['lift'] > 1.8) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
958,"frozenset({50501604, 50502269})",frozenset({50502436}),0.004301,0.054489,0.001370,0.318449,5.844295,1.0,0.001135,1.387293,0.832474,0.023854,0.279172,0.171793,2
959,"frozenset({50501604, 50502436})",frozenset({50502269}),0.003271,0.088677,0.001370,0.418694,4.721569,1.0,0.001080,1.567718,0.790793,0.015122,0.362130,0.217070,2
960,"frozenset({50502269, 50502436})",frozenset({50501604}),0.016375,0.013915,0.001370,0.083645,6.011338,1.0,0.001142,1.076096,0.847526,0.047362,0.070715,0.091042,2
964,"frozenset({50503297, 50502269})",frozenset({50502436}),0.004419,0.054489,0.001264,0.286102,5.250659,1.0,0.001023,1.324435,0.813141,0.021932,0.244961,0.154652,2
965,"frozenset({50503297, 50502436})",frozenset({50502269}),0.002947,0.088677,0.001264,0.428939,4.837095,1.0,0.001003,1.595842,0.795609,0.013991,0.373371,0.221598,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1373,"frozenset({50503441, 62816594})",frozenset({50550228}),0.005358,0.020380,0.001630,0.304209,14.927084,1.0,0.001521,1.407924,0.938034,0.067614,0.289734,0.192096,2
1374,"frozenset({50550228, 62816594})",frozenset({50503441}),0.005325,0.036397,0.001630,0.306086,8.409683,1.0,0.001436,1.388649,0.885807,0.040657,0.279876,0.175435,2
1378,"frozenset({54550994, 50503441})",frozenset({62816594}),0.005349,0.029026,0.001044,0.195229,6.725898,1.0,0.000889,1.206521,0.855899,0.031332,0.171171,0.115604,2
1379,"frozenset({54550994, 62816594})",frozenset({50503441}),0.003755,0.036397,0.001044,0.278096,7.640645,1.0,0.000908,1.334807,0.872397,0.026704,0.250828,0.153394,2


In [46]:
rules["consequents_len"] = rules["consequents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
0,frozenset({50264481}),frozenset({50502269}),0.007201,0.088677,0.002082,0.289174,3.260977,1.0,0.001444,1.282061,0.698372,0.022201,0.220006,0.156328,1,1
1,frozenset({50502269}),frozenset({50264481}),0.088677,0.007201,0.002082,0.023483,3.260977,1.0,0.001444,1.016673,0.760810,0.022201,0.016400,0.156328,1,1
2,frozenset({50264481}),frozenset({50502436}),0.007201,0.054489,0.002134,0.296280,5.437444,1.0,0.001741,1.343590,0.822009,0.035824,0.255725,0.167718,1,1
3,frozenset({50502436}),frozenset({50264481}),0.054489,0.007201,0.002134,0.039156,5.437444,1.0,0.001741,1.033257,0.863121,0.035824,0.032187,0.167718,1,1
4,frozenset({50689433}),frozenset({50264481}),0.032198,0.007201,0.001004,0.031179,4.329787,1.0,0.000772,1.024750,0.794627,0.026147,0.024152,0.085295,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1379,"frozenset({54550994, 62816594})",frozenset({50503441}),0.003755,0.036397,0.001044,0.278096,7.640645,1.0,0.000908,1.334807,0.872397,0.026704,0.250828,0.153394,2,1
1380,"frozenset({50503441, 62816594})",frozenset({54550994}),0.005358,0.061543,0.001044,0.194906,3.167004,1.0,0.000715,1.165650,0.687930,0.015858,0.142109,0.105938,2,1
1381,frozenset({54550994}),"frozenset({50503441, 62816594})",0.061543,0.005358,0.001044,0.016970,3.167004,1.0,0.000715,1.011812,0.729116,0.015858,0.011674,0.105938,1,2
1382,frozenset({50503441}),"frozenset({54550994, 62816594})",0.036397,0.003755,0.001044,0.028693,7.640645,1.0,0.000908,1.025675,0.901949,0.026704,0.025032,0.153394,1,2


In [49]:
rules[ (rules['antecedent_len'] == 1) & 
       (rules['consequents_len'] == 1) &
       (rules['support'] >= 0.005) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
60,frozenset({50502269}),frozenset({50502436}),0.088677,0.054489,0.016375,0.184664,3.389015,1.0,0.011543,1.159658,0.773523,0.129153,0.137676,0.242596,1,1
61,frozenset({50502436}),frozenset({50502269}),0.054489,0.088677,0.016375,0.300528,3.389015,1.0,0.011543,1.302872,0.745553,0.129153,0.232465,0.242596,1,1
72,frozenset({50503441}),frozenset({50502269}),0.036397,0.088677,0.009737,0.267528,3.016878,1.0,0.006510,1.244174,0.693783,0.084424,0.196254,0.188666,1,1
73,frozenset({50502269}),frozenset({50503441}),0.088677,0.036397,0.009737,0.109805,3.016878,1.0,0.006510,1.082463,0.733584,0.084424,0.076181,0.188666,1,1
84,frozenset({50550228}),frozenset({50502269}),0.020380,0.088677,0.006243,0.306352,3.454697,1.0,0.004436,1.313812,0.725321,0.060725,0.238856,0.188379,1,1
85,frozenset({50502269}),frozenset({50550228}),0.088677,0.020380,0.006243,0.070406,3.454697,1.0,0.004436,1.053815,0.779679,0.060725,0.051067,0.188379,1,1
96,frozenset({50652534}),frozenset({50502269}),0.024788,0.088677,0.007079,0.285569,3.220325,1.0,0.004880,1.275592,0.706997,0.066537,0.216050,0.182697,1,1
97,frozenset({50502269}),frozenset({50652534}),0.088677,0.024788,0.007079,0.079824,3.220325,1.0,0.004880,1.059811,0.756562,0.066537,0.056436,0.182697,1,1
102,frozenset({50689433}),frozenset({50502269}),0.032198,0.088677,0.007034,0.218472,2.463686,1.0,0.004179,1.166079,0.613869,0.061791,0.142425,0.148899,1,1
103,frozenset({50502269}),frozenset({50689433}),0.088677,0.032198,0.007034,0.079326,2.463686,1.0,0.004179,1.051188,0.651914,0.061791,0.048696,0.148899,1,1


In [ ]:
# 53482131 = id of Kettle Chips Sea Salt And Black Pepper Corns 150G
# probably need to first conver the item ids into names

# This atleast does not seem to be working, but could also just be because I have not actually set the min support low enough originally?
rules[ (rules['antecedents'] == frozenset({53482131})) ]
      # (rules['lift'] >= 1.05) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
